# MET experiment v10: multi-hop component (GPU, Kaggle)
**Before running:** Settings → Accelerator = **GPU T4 x2**, Internet = **On**; Add Input → your **met-experiment** dataset (**latest version**); HF_TOKEN secret ticked for gated models.

Choose the model **and which test sets to run** in Step 2. Run cells top to bottom through **Step 4**; if it prints `ALL CHECKS PASSED`, click **Save Version → Save & Run All**, then stop the interactive session.

**Resuming a run that was cut off:** in the right panel, Add Input → *Your Work* → *Notebook Output* → pick the earlier version of this notebook. Step 5 finds its partial results file and continues from where it stopped. Runs stop by themselves before the 12 h limit (or `HOURS_BUDGET`), so results are always saved.

## Step 1: environment

In [ ]:
import time
NB_T0 = time.time()   # Kaggle kills a session at 12 h from here; the runs stop themselves before that
import torch, subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],capture_output=True,text=True).stdout)
print('torch', torch.__version__, '| GPUs:', torch.cuda.device_count())

In [ ]:
!pip install -q transformers==4.46.3 accelerate==1.1.1 spacy
!python -m spacy download en_core_web_sm -q

In [ ]:
import glob, os, shutil
# pick the dataset copy that contains the NaN guard (newest code)
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/**/met/methods.py', recursive=True)
         if 'FloatingPointError' in open(p).read() and os.path.exists(os.path.join(os.path.dirname(os.path.dirname(p)),'data','met_multihop.jsonl')) and 'robustrag_decoding' in open(p).read() and 'grounded' in open(p).read() and os.path.exists(os.path.join(os.path.dirname(os.path.dirname(p)),'met','madam.py')) and 'AGG_PREFILL' in open(os.path.join(os.path.dirname(os.path.dirname(p)),'met','madam.py')).read() and 'max_hours' in open(os.path.join(os.path.dirname(os.path.dirname(p)),'run.py')).read()]
assert cands, 'OLD DATASET VERSION ATTACHED (needs the multi-hop version with data/met_multihop.jsonl). Upload the newest met-experiment.zip as a NEW VERSION of the dataset, then in the right panel remove the old input and add the latest version.'
shutil.copytree(cands[0], '/kaggle/working/exp', dirs_exist_ok=True)
os.chdir('/kaggle/working/exp'); os.makedirs('results', exist_ok=True)
print('code from', cands[0]); print(os.listdir('.'))

## Step 2: choose the model
Qwen2.5 and Gemma-2 **must** use `bfloat16` (float16 overflows). Llama-3.1 and Mistral are fine in either.

In [ ]:
# PRESET for the multi-hop runs: QWEN. Nothing to edit -- commit as is.
MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
DTYPE = 'bfloat16'
RUNS = ['multihop']
HOURS_BUDGET = 2.3   # stops cleanly and saves when reached; lower or raise to fit your weekly quota
TAG = MODEL_ID.split('/')[-1]
# short tag for the per-model adaptive-attack data (the attack depends on this model's prior)
ATAG = {'Qwen2.5-7B-Instruct':'q7','Llama-3.1-8B-Instruct':'l8','Mistral-7B-Instruct-v0.3':'m7'}[TAG]
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN'); print('HF token loaded')
except Exception:
    print('no HF token (fine for Qwen / Mistral)')
print('=' * 60); print('MODEL:', TAG, '| DTYPE:', DTYPE, '| RUNS:', RUNS, '| HOURS_BUDGET:', HOURS_BUDGET); print('=' * 60)
# fail fast, before the model download, if any run's data file is missing
import os as _os
_need = {'adaptive': f'data/met_adaptive_{ATAG}.jsonl', 'adaptive_ret': f'data/met_adaptive_ret_{ATAG}.jsonl',
         'edits': 'data/met_edits.jsonl', 'cf': 'data/met.jsonl',
         'madam': 'data/met_madam.jsonl', 'multihop': 'data/met_multihop.jsonl', 'madam_adaptive': f'data/met_madam_adaptive_{ATAG}.jsonl'}
for _r in RUNS:
    if _r in _need:
        assert _os.path.exists(_need[_r]), f'{_need[_r]} missing: the attached dataset version is too old'
print('data files present for', RUNS)

## Step 3: download the model (auto-retries on stalls)

In [ ]:
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '60'
from huggingface_hub import snapshot_download
MODEL = None
for attempt in range(20):
    try:
        MODEL = snapshot_download(MODEL_ID, allow_patterns=['*.json','*.safetensors','*.txt','*.model'], max_workers=1,
                                  ignore_patterns=['consolidated*', 'original/*'])  # skip duplicate Mistral/Meta-format weights
        break
    except Exception as e:
        print(f'attempt {attempt+1} failed ({type(e).__name__}), retrying...')
assert MODEL, 'download failed 20 times'
print('MODEL =', MODEL)

## Step 4: safety check (1 fact x 4 poisoning levels)

In [ ]:
for f in glob.glob(f'results/{TAG}_smoke.jsonl'): os.remove(f)
r = subprocess.run(['python','run.py','--model',MODEL,'--data','data/met_known.jsonl','--limit_q','2',
                    '--out',f'results/{TAG}_smoke.jsonl','--dtype',DTYPE,'--device','auto'],capture_output=True,text=True)
print(r.stdout[-2500:]); print(r.stderr[-1500:] if r.returncode else '')
assert r.returncode == 0, 'run.py crashed - see error above'
import json, math
rows = [json.loads(l) for l in open(f'results/{TAG}_smoke.jsonl')]
bad_conf = [x['id'] for x in rows if not math.isfinite(x['prior_conf'])]
answers = [a for x in rows for a in list(x['pred'].values()) + (x['isolated'] or [])]
bang = sum(a.count('!') for a in answers)
print('sample answers:', answers[:8])
print('prior_conf values:', [x['prior_conf'] for x in rows])
assert not bad_conf, f'NaN confidence in {bad_conf}'
assert bang <= 2, f'{bang} stray "!" characters in answers -> precision problem'
print('ALL CHECKS PASSED')

## Step 5: full runs
Float16 7-8B model: known ≈ 45 min, met ≈ 1 h, retrieval ≈ 2 h; RobustRAG-only runs: rrag_known ≈ 40 min, rrag_retrieval ≈ 1.5 h (about 4x longer in bfloat16).

In [ ]:
SPECS = {'known':     ['--data','data/met_known.jsonl'],
         'met':       ['--data','data/met.jsonl','--limit_q','30'],
         'retrieval': ['--data','data/met_retrieval.jsonl','--parts','retrieval'],
         # all 300 Faithfulness-QA counterfactual pairs, main methods + RobustRAG (legitimate-edit test)
         'cf':        ['--data','data/met.jsonl','--parts','counterfactual',
                       '--methods','closed_book,rag,cad,adacad,arr,sdag,vote,pvote,rrag_kw,rrag_dec'],
         # legitimate edits inside a 5-document context (200 questions x orig/edit/update), all methods
         'edits':     ['--data','data/met_edits.jsonl',
                       '--methods','closed_book,rag,cad,adacad,arr,sdag,vote,pvote,rrag_kw,rrag_dec'],
         # smaller real-retrieval run (35 questions per corpus = 105 x 4 injection levels), all methods (for slow bf16 models)
         'retrieval_small': ['--data','data/met_retrieval.jsonl','--parts','retrieval','--limit_q','35',
                       '--methods','closed_book,rag,cad,adacad,arr,sdag,vote,pvote,rrag_kw,rrag_dec'],
         # adaptive attack A1: the attacker targets THIS model's own closed-book answer
         'adaptive':  ['--data',f'data/met_adaptive_{ATAG}.jsonl',
                       '--methods','closed_book,rag,cad,adacad,arr,sdag,vote,pvote,rrag_kw,rrag_dec'],
         # adaptive attack A1-R: the same attack injected into the real BEIR corpus and re-retrieved
         'adaptive_ret': ['--data',f'data/met_adaptive_ret_{ATAG}.jsonl','--parts','retrieval',
                       '--methods','closed_book,rag,cad,adacad,arr,sdag,vote,pvote,rrag_kw,rrag_dec'],
         # MADAM-RAG baseline on a fixed paired subset (it costs ~20-40x the other methods)
         'madam':     ['--data','data/met_madam.jsonl','--methods','closed_book,madam'],
         'madam_adaptive': ['--data',f'data/met_madam_adaptive_{ATAG}.jsonl','--methods','closed_book,madam'],
         # multi-hop component: 100 HotpotQA bridge questions x {orig, edit, poison1, poison2}, all main methods
         'multihop':  ['--data','data/met_multihop.jsonl','--parts','multihop',
                       '--methods','closed_book,rag,cad,adacad,arr,sdag,vote,pvote,rrag_kw,rrag_dec']}
RR = ['--methods','closed_book,rrag_kw,rrag_dec']
SPECS.update({'rrag_known': SPECS['known']+RR, 'rrag_met': SPECS['met']+RR, 'rrag_retrieval': SPECS['retrieval']+RR})
for name in RUNS:
    print(f'===== {name} =====', flush=True)
    out = f'results/{TAG}_{name}' + ('_agg2' if name.startswith('madam') else '') + '.jsonl'
    # resume: pick up a partial results file from an attached earlier notebook output, if any
    prev = [p for p in glob.glob(f'/kaggle/input/**/' + os.path.basename(out), recursive=True)]
    if prev and not os.path.exists(out):
        best = max(prev, key=lambda p: sum(1 for _ in open(p)))
        shutil.copy(best, out); print('RESUMING from', best, '->', sum(1 for _ in open(out)), 'instances already done', flush=True)
    left_h = HOURS_BUDGET - (time.time() - NB_T0) / 3600 - 0.25   # 15 min reserve for model load + saving
    assert left_h > 0.3, 'no time left in the budget for ' + name
    print(f'time budget for this run: {left_h:.2f} h', flush=True)
    cmd = ['python','run.py','--model',MODEL,'--out',out,'--dtype',DTYPE,'--device','auto','--max_hours',f'{left_h:.3f}'] + SPECS[name]
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if 'past_key_values' not in line: print(line, end='', flush=True)
    assert p.wait() == 0, f'{name} run failed'

In [ ]:
for name in RUNS:
    subprocess.run(['python','analyze.py',f'results/{TAG}_{name}' + ('_agg2' if name.startswith('madam') else '') + '.jsonl'])
for f in glob.glob(f'results/{TAG}_*.jsonl'): shutil.copy(f, '/kaggle/working/')
print(sorted(os.listdir('/kaggle/working')))
for name in RUNS:
    n = sum(1 for _ in open(f'results/{TAG}_{name}' + ('_agg2' if name.startswith('madam') else '') + '.jsonl'))
    print(f'{name}: {n} instances saved')